# AUDUSD/NZDUSD — independent-day strategies

Four strategies on the cointegration spread, fit and back-tested **independently per trading day** on three months: **2024-08, 2024-09, 2025-08**.

* Bars: 200-tick pre-averaged (mid = mean of N tick mids)
* Per-day rolling cointegration ($W_eta=50$), per-day rolling $z$-score ($W_z=25$)
* Per-day MS-AR(1) on the spread, $K=2$, multi-seed EM
* Strategies: Buy & Hold, Baseline, AR(1) gate, MS-AR(1) dynamic threshold

The OOS forward filter is intentionally deferred — we use the in-sample smoothed posteriors $\gamma_t^{\mathrm{MR}}$ directly inside the strategies, so this is a *best-case* picture of how the regime information would be used if it were known.


In [1]:
# Colab bootstrap (silently skipped on local environments).
import importlib.util, sys, os
if importlib.util.find_spec('google.colab') is not None:
    import subprocess
    subprocess.run(
        ['curl', '-sL',
         'https://raw.githubusercontent.com/egil10/stk-mat2011/main/code/scripts/colab.py',
         '-o', '/content/colab.py'],
        check=True,
    )
    sys.path.insert(0, '/content')
    from colab import setup
    setup('code/strats')
else:
    sys.path.insert(0, os.path.abspath('../scripts'))


Mounted at /content/drive
CWD           : /content/stk-mat2011/code/strats
Scripts path  : /content/stk-mat2011/code/scripts
Data symlink  : /content/stk-mat2011/code/data/processed -> /content/drive/MyDrive/GITHUB-COPILOT/stk-mat2011/data/processed
Parquet files : 916


In [2]:
%pip install --quiet arch statsmodels numba optuna


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.3/981.3 kB 13.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 17.6 MB/s eta 0:00:00


In [3]:
import pandas as pd

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

PAIR_A, PAIR_B = 'AUDUSD', 'NZDUSD'
MONTHS         = ['202408', '202409', '202508']
DATA_DIR       = '../data/processed'

# Bars: tick-pre-averaged (Jacod-style mid = mean of N tick mids).
# 200 ticks/bar gives ~250-1000 bars/day across the three pairs.
BAR_CFG = dict(
    agg_type     = 'tick',
    threshold    = 200,
    price_agg    = 'mean',
    active_hours = (0, 24),
)

# Per-day fit: rolling beta + rolling z + MS-AR(1) on the day's spread.
FIT_CFG = dict(
    coint_window  = 50,
    z_window      = 25,
    k_regimes     = 2,
    winsorize_std = 4.0,
    scaling       = 10000,
    n_init        = 3,
)

# Strategies (paper section on regime-aware trading).
STRAT_CFG = dict(
    z_quiet          = 1.3,
    z_volatile       = 2.5,
    exit_z           = 0.0,
    danger_threshold = 0.30,
    fee_bps          = 0.5,
    slippage_mode    = 'half_spread',
    flatten_eod      = True,
    prob_smoothing   = 0,
)


## Per-month run

For each month we

1. Aggregate raw bid/ask ticks into 200-tick pre-averaged bars (`SPREAD`).
2. Fit each trading day independently — rolling cointegration ($eta_t, lpha_t$), rolling $z$-score, MS-AR(1) on the day's spread (`ENGINE.each_day`).
3. Run Buy & Hold, Baseline, AR(1) and MS-AR(1) through the same back-test machinery (`BACKTESTER`).
4. Score with `TEARSHEET`.


In [4]:
def files(pair, month):
    p = pair.lower()
    return (
        [f'{DATA_DIR}/{p}_dukascopy_ask_{month}.parquet'],
        [f'{DATA_DIR}/{p}_dukascopy_bid_{month}.parquet'],
    )

runs = {}
for m in MONTHS:
    print(f'
=== {m}  {PAIR_A}/{PAIR_B} ===')

    ask_a, bid_a = files(PAIR_A, m)
    ask_b, bid_b = files(PAIR_B, m)

    df = SPREAD(**BAR_CFG).build([ask_a, bid_a, ask_b, bid_b], verbose=False)
    n_bars = len(df)
    n_days = df.index.normalize().unique().shape[0]
    print(f'  built {n_bars:,} bars over {n_days} days')

    fitted, params = ENGINE.each_day(df, **FIT_CFG, verbose=False)
    bt = BACKTESTER(fitted).run(**STRAT_CFG)
    runs[m] = (bt, params, fitted)

    pnl = {s: bt[f'Return_{s}'].sum() * 1e4 for s in ['BuyHold', 'Baseline', 'AR', 'MS_AR']}
    print(
        f'  net pnl (bps):  BH={pnl["BuyHold"]:+8.1f}  '
        f'Base={pnl["Baseline"]:+8.1f}  '
        f'AR={pnl["AR"]:+8.1f}  '
        f'MS-AR={pnl["MS_AR"]:+8.1f}'
    )


SyntaxError: unterminated f-string literal (detected at line 10) (330745291.py, line 10)

## Per-month tearsheets


In [ ]:
for m, (bt, params, _) in runs.items():
    print(f'
{"="*28}  {m}  {"="*28}')
    ts = TEARSHEET(bt, df_params=params)
    ts.generate_report()
    ts.plot_performance()
    ts.plot_positions_and_regimes()
    ts.plot_markov_dynamics()


## Cross-month summary


In [ ]:
ANN = 252 * 24 * 60   # tick-clock annualisation, matches MONTH default
rows = []
for m, (bt, _, _) in runs.items():
    row = {'Month': m}
    for s in ['BuyHold', 'Baseline', 'AR', 'MS_AR']:
        r = bt[f'Return_{s}'].fillna(0)
        sd = r.std()
        row[f'{s}_PnL_bps'] = float(r.sum() * 1e4)
        row[f'{s}_Sharpe']  = float(r.mean() / sd * ANN**0.5) if sd > 0 else 0.0
        row[f'{s}_Trades']  = int((bt[f'Target_{s}'].diff().abs() > 0).sum() / 2)
    rows.append(row)

summary = pd.DataFrame(rows).set_index('Month')

print('
=== PnL bps by month ===')
print(summary[[c for c in summary.columns if c.endswith('PnL_bps')]].round(1).to_string())

print('
=== Sharpe (tick-clock annualised) ===')
print(summary[[c for c in summary.columns if c.endswith('Sharpe')]].round(2).to_string())

print('
=== Trades (per round-trip) ===')
print(summary[[c for c in summary.columns if c.endswith('Trades')]].to_string())
